In [1]:
# import required packages
from sklearn.cluster import DBSCAN
from sklearn.cluster import HDBSCAN
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import folium
import os
import yaml
from pprint import pprint
import pandas as pd
import geopandas as gpd
from scipy.spatial import ConvexHull
import numpy as np

from pyproj import Transformer

sns.set_theme(context='paper', style='darkgrid')

In [2]:
INDO_CRS = "EPSG:23867"
LL_CRS = "EPSG:4326"

def xy_to_ll(xy):
        return Transformer.from_crs(INDO_CRS, LL_CRS).transform(xy[0], xy[1])

def ll_to_xy(ll):
    return Transformer.from_crs(LL_CRS, INDO_CRS).transform(ll[0], ll[1])

In [ ]:
# construct dataset for intermediaries
dfs = []
root = '../FactoredPlatformSolver/data/instances'

for f in os.listdir(root):
    if f.endswith('.yaml') and not f.startswith('aggregate'):
        with open(os.path.join(root, f), 'r') as file:
            data = yaml.safe_load(file)
            df_i = pd.DataFrame(data['intermediaries']).drop(['capacity', 'routes'], axis=1)
            dfs.append(df_i)

ints_df = pd.concat(dfs)

ints_df['int_lat'] = ints_df.location.apply(lambda x: x[0])
ints_df['int_lon'] = ints_df.location.apply(lambda x: x[1])

ints_df['int_x'], ints_df['int_y'] = ll_to_xy([ints_df.int_lat.values, ints_df.int_lon.values])

ints_df.drop('location', axis=1, inplace=True)

ints_df.rename(
    columns={'id': 'int_id'},
    inplace=True
)

ints_df.drop_duplicates(inplace=True)

ints_df.reset_index(drop=True, inplace=True)

ints_df.to_csv('ints_2.csv', index=False)

ints_df


In [ ]:
# construct master dataset for mills
dfs = []

for f in os.listdir(root):
    if f.endswith('.yaml') and not f.startswith('aggregate'):
        with open(os.path.join(root, f), 'r') as file:
            data = yaml.safe_load(file)
            df_m = pd.DataFrame(data['mills'])
            dfs.append(df_m)

mills_df = pd.concat(dfs)

mills_df['mill_lat'] = mills_df.location.apply(lambda x: x[0])
mills_df['mill_lon'] = mills_df.location.apply(lambda x: x[1])

mills_df['mill_x'], mills_df['mill_y'] = ll_to_xy([mills_df.mill_lat.values, mills_df.mill_lon.values])

mills_df.drop('location', axis=1, inplace=True)

mills_df.rename(
    columns={'id': 'mill_id'},
    inplace=True
)

mills_df.drop_duplicates(inplace=True)

mills_df.reset_index(drop=True, inplace=True)

mills_df.to_csv('mills_2.csv', index=False)

mills_df

In [ ]:
# construct master dataset for all instances

dfs = []

for f in os.listdir(root):
    if f.endswith('.yaml') and not f.startswith('aggregate'):
        with open(os.path.join(root, f), 'r') as file:
            data = yaml.safe_load(file)
        date = f.split('.')[0]

        farmers = pd.DataFrame(data['farmers']).set_index('id')

        ints = data['intermediaries']


        pickup_data = []

        for i in ints:

            if len(i['routes']) == 0:
                continue

            route = i['routes'][0]

            f_data = farmers.loc[route].reset_index()

            pickup_data.append(f_data)

        daily_df = pd.concat(pickup_data)

        daily_df['date'] = pd.to_datetime(date)
        dfs.append(daily_df)
        
farmers_df = pd.concat(dfs)

farmers_df.rename(
    columns={'id': 'farmer_id',
            'intermediary': 'int_id',
            'location': 'farmer_loc'},
    inplace=True
)

farmers_df = farmers_df.merge(ints_df, on="int_id", how="left")

farmers_df['farmer_lat'] = farmers_df['farmer_loc'].apply(lambda loc: loc[0])
farmers_df['farmer_lon'] = farmers_df['farmer_loc'].apply(lambda loc: loc[1])
farmers_df.drop('farmer_loc', axis=1, inplace=True)

farmers_df['farmer_x'], farmers_df['farmer_y'] = ll_to_xy([farmers_df.farmer_lat.values, farmers_df.farmer_lon.values])

farmers_df['offset_x'] = farmers_df['farmer_x'] - farmers_df['int_x']
farmers_df['offset_y'] = farmers_df['farmer_y'] - farmers_df['int_y']

farmers_df['distance'] = (farmers_df['offset_x'] ** 2 + farmers_df['offset_y'] ** 2) ** 0.5
farmers_df.drop(['offset_x', 'offset_y'], axis=1, inplace=True)

farmers_df.to_csv('farmers_2.csv', index=False)

farmers_df

In [7]:
f_df = pd.read_csv('data/farmers.csv')
f2_df = pd.read_csv('data/farmers_2.csv')

In [12]:
len(f_df)

2472

In [15]:
print(len(f_df.drop_duplicates(['farmer_lat', 'farmer_lon'])))
print(len(f2_df.drop_duplicates(['farmer_lat', 'farmer_lon'])))

554
233
